---

### What we'll do in this notebook

1. Get an API key from FRED (Federal Reserve Economic Data)
2. Fetch data through the API (wrapped in a class)
3. Turn it into a table
4. Clean the table until it's tidy
5. Let the user choose the output shape: **long** or **wide**
6. Save the result

FRED provides economic time series (GDP, unemployment, inflation, interest
rates, and thousands more), each identified by a `series_id`. We'll fetch a
handful of series, combine them, clean them, and save the result in a shape
the user can choose.

---

# Part 1: Getting an API Key

### Steps to create an API key on FRED

1. Go to [fred.stlouisfed.org/docs/api/api_key.html](https://fred.stlouisfed.org/docs/api/api_key.html)
2. Register for a free account, then request an API key
3. Copy the API key

### Storing the API key safely

Never write the key directly inside the notebook. Store it in a
`.env` file in the same folder as this notebook:

```
FRED_API_KEY=your_api_key_here
```

Save the file, then move on to the next cell.

In [1]:
# !pip install requests python-dotenv --quiet

In [2]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()  # reads the .env file

API_KEY = os.getenv("FRED_API_KEY")

if API_KEY:
    print("API key loaded successfully.")
else:
    print("API key not found. Make sure the .env file exists and is filled in correctly.")

API key loaded successfully.


If you see "API key not found", check two things:

1. The `.env` file is in the same folder as this notebook, not somewhere else
2. There are no quotes or spaces around the key inside the `.env` file

Don't move on to the next cell until "loaded successfully" appears.

---

# Part 2: Trying the API With One Small Call

### Understanding the endpoint and parameters

FRED has an endpoint called `series/observations`, located at:

```
https://api.stlouisfed.org/fred/series/observations
```

Parameters we need to pass through `params`:

- `series_id` : the official code for a data series (e.g. `GDP`, `UNRATE`, `CPIAUCSL`)
- `api_key` : our identity card
- `file_type` : the response format we want, set to `json` (defaults to XML if omitted)

### Why try one small call first

Before writing a full class around it, we make one simple call to see
exactly what the response looks like — what keys it has, how the data for a
single observation is structured, and what data types come back.

In [3]:
api_url = "https://api.stlouisfed.org/fred/series/observations"

params = {
    "series_id" : "GDP",
    "api_key"   : API_KEY,
    "file_type" : "json"
}

response = requests.get(api_url, params=params)
print(f"Status Code: {response.status_code}")

result = response.json()
print("Top-level keys in the response:", list(result.keys()))
print(f"Number of observations returned: {len(result['observations'])}")

Status Code: 200
Top-level keys in the response: ['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations']
Number of observations returned: 322


In [4]:
# Look at the shape of a single observation
first_observation = result["observations"][0]
first_observation

{'realtime_start': '2026-09-17',
 'realtime_end': '2026-09-17',
 'date': '1946-01-01',
 'value': '.'}

Notice the shape. A single observation is a dictionary with keys like
`date`, `value`, and a couple of `realtime_start`/`realtime_end` metadata
fields we can ignore. The structure is flat — no nested dictionaries — so we
can grab `["date"]` and `["value"]` directly.

One important thing to check: the data type of `value`.

In [5]:
print("Date  :", first_observation["date"])
print("Value :", first_observation["value"])
print("Python type of value:", type(first_observation["value"]))

Date  : 1946-01-01
Value : .
Python type of value: <class 'str'>


`value` comes back as **text** (a string), not a number, even though it
looks like a number. Keep this in mind for Part 4, since we'll need to
convert it to a numeric type.

FRED also has a quirk worth noting: when a value isn't available yet for a
given date, it isn't empty/`null` — it's the **literal string `"."`**. We'll
handle this in Part 4 as well.

---

# Part 3: Wrapping It in a Class

### Why wrap it in a class

Wrapping the fetch logic in a class keeps it tidy and reusable across
different `series_id` values, without repeating the request/retry code
every time we want another series.

- **The ID card it carries around**, the API key, stored as `self.api_key`
- **The capability it can perform**, fetching data for one series, becomes a **method**

### A note on date ranges

If `observation_start`/`observation_end` aren't passed, FRED defaults to
returning the **entire available history** of a series — for `GDP` that
means all the way back to 1947, which can be a lot of rows. The class below
accepts optional `start_date`/`end_date` arguments so the range can be
limited when needed. Leave them as `None` to get the full history.

In [6]:
class FREDClient:

    def __init__(self, api_key):
        # The ID card is stored here so every method below can use it
        self.api_key = api_key
        self.api_url = "https://api.stlouisfed.org/fred/series/observations"

    def fetch_series(self, series_id, start_date=None, end_date=None):
        params = {
            "series_id" : series_id,
            "api_key"   : self.api_key,
            "file_type" : "json"
        }
        if start_date:
            params["observation_start"] = start_date
        if end_date:
            params["observation_end"] = end_date

        try:
            # Main plan: contact FRED
            # timeout=20 means we only wait up to 20 seconds
            response = requests.get(self.api_url, params=params, timeout=20)
        except Exception:
            # Backup plan: if the connection has a problem, try once more
            print("Connection issue, retrying...")
            time.sleep(3)
            response = requests.get(self.api_url, params=params, timeout=20)

        if response.status_code != 200:
            print(f"Failed to fetch data for '{series_id}'. Status: {response.status_code}")
            return pd.DataFrame()

        observations = response.json()["observations"]

        data = []
        for obs in observations:
            data.append({
                "date"      : obs["date"],
                "value"     : obs["value"],
                "series_id" : series_id
            })

        return pd.DataFrame(data)


print("FREDClient class is ready to use!")

FREDClient class is ready to use!


Notice the `series_id` column added manually (not from the API) on every
row. That's what lets us, after concatenating several series into one big
table, still tell which series a given row belongs to.

### Why fetch several series at once

We fetch a few related `series_id` values (GDP, unemployment rate,
inflation), then combine the results with `pd.concat` into one table. We'll
also limit the date range here using the `start_date`/`end_date` arguments,
so we don't pull decades of history we don't need.

In [7]:
client = FREDClient(API_KEY)

series_list = ["GDP", "UNRATE", "CPIAUCSL"]

# Limit the range instead of pulling full history back to the 1940s.
# Set these to None if you actually want the full available history.
START_DATE = "2015-01-01"
END_DATE = None   # None means "up to the most recent observation"

all_tables = []

for series_id in series_list:
    table = client.fetch_series(series_id, start_date=START_DATE, end_date=END_DATE)
    print(f"Series '{series_id}': {len(table)} rows")
    all_tables.append(table)
    time.sleep(1)

df_fred = pd.concat(all_tables, ignore_index=True)

print()
print(f"Total rows collected: {len(df_fred)}")
df_fred.head()

Series 'GDP': 46 rows
Series 'UNRATE': 140 rows
Series 'CPIAUCSL': 140 rows

Total rows collected: 326


,date,value,series_id
0,2015-01-01,18063.529,GDP
1,2015-04-01,18279.784,GDP
2,2015-07-01,18401.626,GDP
3,2015-10-01,18435.137,GDP
4,2016-01-01,18525.933,GDP


If your assignment requires a minimum number of rows, check the number
above. GDP is quarterly data (relatively few rows), while UNRATE and
CPIAUCSL are monthly (more rows) — so the combined total is usually
generous, but check it yourself.

At this point, `df_fred` is in **long format**: each row is one
`(date, series_id, value)` observation. This is the natural shape the data
comes in, and it's also one of the two output options we'll offer the user
in Part 5.

---

# Part 4: Cleaning the Data

Three things to handle:

1. Missing / not-yet-available values
2. Duplicate rows
3. Incorrect data types

In [8]:
print("1. Number of empty cells per column:")
print(df_fred.isnull().sum())
print()

print("2. Suspicious values in the 'value' column (FRED's missing-data marker):")
print(df_fred[df_fred["value"] == "."].shape[0], "rows marked with a dot")
print()

print("3. Number of duplicate rows (based on date + series_id):")
print(df_fred.duplicated(subset=["date", "series_id"]).sum())
print()

print("4. Data type of each column:")
print(df_fred.dtypes)

1. Number of empty cells per column:
date         0
value        0
series_id    0
dtype: int64

2. Suspicious values in the 'value' column (FRED's missing-data marker):
2 rows marked with a dot

3. Number of duplicate rows (based on date + series_id):
0

4. Data type of each column:
date         object
value        object
series_id    object
dtype: object


### Reading the results

`isnull().sum()` above likely shows 0 for the `value` column, even though
some data really is missing. That's because FRED doesn't use `None`/`NaN`
to mark missing data — it uses the **literal string `"."`** — so pandas'
built-in `isnull()` doesn't catch it. That's why check #2 above is written
manually.

Duplicates are checked using the combination of `date` + `series_id`,
because the same date legitimately appears across many different series —
that's not a duplicate, it's just data from another series.

The `date` and `value` columns are likely still `object` (text) type,
even though they should be a date and a number respectively.

### Handling missing values (the dot marker)

A small helper function that handles this one specific problem: turning
FRED's `"."` marker into a proper missing value.

In [9]:
def clean_value(text):
    if text == ".":
        return None
    return text


df_fred["value"] = df_fred["value"].apply(clean_value)

print("Rows with a missing value (after marking as None):")
print(df_fred["value"].isnull().sum())

Rows with a missing value (after marking as None):
2


### Handling duplicate rows

In [10]:
rows_before = len(df_fred)

df_clean = df_fred.drop_duplicates(subset=["date", "series_id"])

print(f"Rows before  : {rows_before}")
print(f"Rows after   : {len(df_clean)}")
print(f"Duplicates removed: {rows_before - len(df_clean)}")

Rows before  : 326
Rows after   : 326
Duplicates removed: 0


### Handling data types

The `date` column is converted to a real date type. The `value` column is
converted to a numeric type using `pd.to_numeric`, with
`errors="coerce"` so any problematic value automatically becomes `NaN`
instead of stopping the program with an error.

In [11]:
def to_date(text):
    return pd.to_datetime(text)


def to_number(text):
    return pd.to_numeric(text, errors="coerce")


print("Types before -> date:", df_clean["date"].dtype, "| value:", df_clean["value"].dtype)

df_clean["date"] = df_clean["date"].apply(to_date)
df_clean["value"] = to_number(df_clean["value"])

print("Types after  -> date:", df_clean["date"].dtype, "| value:", df_clean["value"].dtype)
df_clean.head()

Types before -> date: object | value: object
Types after  -> date: datetime64[ns] | value: float64


,date,value,series_id
0,2015-01-01,18063.529,GDP
1,2015-04-01,18279.784,GDP
2,2015-07-01,18401.626,GDP
3,2015-10-01,18435.137,GDP
4,2016-01-01,18525.933,GDP


In [12]:
# Final check before we consider this done

print(f"Row count            : {len(df_clean)}")
print(f"Series still present  : {df_clean['series_id'].unique()}")
print(f"Duplicates remaining  : {df_clean.duplicated(subset=['date', 'series_id']).sum()}")
print(f"'date' column type    : {df_clean['date'].dtype}")
print(f"'value' column type   : {df_clean['value'].dtype}")
print(f"Missing values (NaN)  : {df_clean['value'].isnull().sum()}")

Row count            : 326
Series still present  : ['GDP' 'UNRATE' 'CPIAUCSL']
Duplicates remaining  : 0
'date' column type    : datetime64[ns]
'value' column type   : float64
Missing values (NaN)  : 2


Note: the remaining `NaN` values in `value` are **not** dropped — they're
kept as-is, because they're a valid representation of "data not available
for this date yet", unlike the missing-title rows in the news dataset which
were genuinely useless and worth discarding.

At this point, `df_clean` is our clean **long-format** dataset. This is the
dataset we'll offer to reshape in Part 5.

---

# Part 5: Choosing the Output Format (Long or Wide)

Instead of saving only one fixed shape, we let the user pick between two
common shapes for time-series data.

### Long format (what `df_clean` already looks like)

| date | series_id | value |
|---|---|---|
| 1947-01-01 | GDP | 243.164 |
| 1947-01-01 | UNRATE | 3.9 |

- Easy to add new series without changing the table structure
- Good for `groupby`/aggregation per series
- This is the "tidy data" convention most plotting/analysis libraries expect

### Wide format

| date | GDP | UNRATE | CPIAUCSL |
|---|---|---|---|
| 1947-01-01 | 243.164 | 3.9 | ... |

- Easier to read at a glance and to plot multiple series directly
- Built with `df.pivot(index="date", columns="series_id", values="value")`
- Since each FRED series has its own frequency (GDP is quarterly, UNRATE is
  monthly), a wide table will naturally contain a lot of `NaN` — that's
  expected, not a bug

### Letting the user choose

We wrap the choice in a small function, and expose one variable the user can
change without touching the rest of the notebook.

In [13]:
def reshape_data(df_long, output_format="long"):
    """
    Reshape the cleaned long-format FRED dataset.

    Parameters
    ----------
    df_long : pd.DataFrame
        Cleaned data with columns ['date', 'series_id', 'value'].
    output_format : str
        Either "long" (no change) or "wide" (pivoted, one column per series).

    Returns
    -------
    pd.DataFrame
    """
    if output_format == "long":
        return df_long.copy()

    elif output_format == "wide":
        df_wide = df_long.pivot(index="date", columns="series_id", values="value")
        df_wide = df_wide.reset_index()
        df_wide.columns.name = None
        return df_wide

    else:
        raise ValueError(f"Unknown output_format: '{output_format}'. Use 'long' or 'wide'.")

In [17]:
# --- User-facing option ---
# Change this to "long" or "wide" depending on what you need downstream.
OUTPUT_FORMAT = "wide"   # <- try changing this to "wide" and re-run the cell below

df_output = reshape_data(df_clean, output_format=OUTPUT_FORMAT)

print(f"Output format: {OUTPUT_FORMAT}")
print(f"Shape: {df_output.shape}")
df_output.head()

Output format: wide
Shape: (140, 4)


,date,CPIAUCSL,GDP,UNRATE
0,2015-01-01,234.747,18063.529,5.7
1,2015-02-01,235.342,NaN,5.5
2,2015-03-01,235.976,NaN,5.4
3,2015-04-01,236.222,18279.784,5.4
4,2015-05-01,237.001,NaN,5.6


Try setting `OUTPUT_FORMAT = "wide"` above and re-running that cell — you'll
see the table pivot so each series becomes its own column, with `NaN` filling
in dates where a particular series has no observation.

---

# Part 6: Saving the Result

We save the dataset in whichever format was selected in Part 5, and include
the format name in the filename so it's clear which shape it is.

In [18]:
output_filename = f"dataset_fred_{OUTPUT_FORMAT}.csv"

df_output.to_csv(output_filename, index=False)
print(f"Data saved to: {output_filename}")

df_check = pd.read_csv(output_filename)
print(f"File read back: {len(df_check)} rows, {len(df_check.columns)} columns")
df_check.head()

Data saved to: dataset_fred_wide.csv
File read back: 140 rows, 4 columns


,date,CPIAUCSL,GDP,UNRATE
0,2015-01-01,234.747,18063.529,5.7
1,2015-02-01,235.342,NaN,5.5
2,2015-03-01,235.976,NaN,5.4
3,2015-04-01,236.222,18279.784,5.4
4,2015-05-01,237.001,NaN,5.6


> Once this notebook is moved into the CCDS project structure
> (`notebooks/01_explore_fred.ipynb`), change the save path to
> `"../data/processed/" + output_filename` to follow the project's folder
> convention.

---

# Part 7: Slide Material

In [16]:
print("=" * 50)
print("NUMBERS FOR THE SLIDES")
print("=" * 50)
print(f"Data source      : FRED (Federal Reserve Economic Data)")
print(f"Series used      : {', '.join(series_list)}")
print(f"Output format    : {OUTPUT_FORMAT}")
print()
print(f"Rows before cleaning : {len(df_fred)}")
print(f"Rows after cleaning  : {len(df_clean)}")
print(f"Rows in saved output : {len(df_output)}")
print()
print("Classes built:")
print("  1. FREDClient - fetches time-series data from the FRED API")
print()
print("Functions built:")
print("  1. clean_value   - marks '.' (not yet available) as missing")
print("  2. to_date       - converts text into a real date type")
print("  3. to_number     - converts text into a numeric type")
print("  4. reshape_data  - lets the user pick long or wide output format")
print()
print("Findings from cleaning:")
print(f"  Duplicates removed        : {rows_before - len(df_clean)}")
print(f"  Missing values (NaN)      : {df_clean['value'].isnull().sum()}")
print("=" * 50)

NUMBERS FOR THE SLIDES
Data source      : FRED (Federal Reserve Economic Data)
Series used      : GDP, UNRATE, CPIAUCSL
Output format    : long

Rows before cleaning : 326
Rows after cleaning  : 326
Rows in saved output : 326

Classes built:
  1. FREDClient - fetches time-series data from the FRED API

Functions built:
  1. clean_value   - marks '.' (not yet available) as missing
  2. to_date       - converts text into a real date type
  3. to_number     - converts text into a numeric type
  4. reshape_data  - lets the user pick long or wide output format

Findings from cleaning:
  Duplicates removed        : 0
  Missing values (NaN)      : 2


---

# Summary

| Aspect | Covered in |
|---|---|
| Fetching data via API | Part 2 and 3 |
| OOP structure | Part 3, `FREDClient` class |
| Functions and modularity | Part 4-5, `clean_value`, `to_date`, `to_number`, `reshape_data` |
| Data cleaning | Part 4 |
| User-selectable output shape (long/wide) | Part 5 |
| Saving the result | Part 6 |

### Things worth remembering about the FRED API specifically

1. One `series_id` = one full time series; there's no free-text keyword search
2. Without `observation_start`/`observation_end`, a request returns the entire available history
3. Missing data is marked with the text `"."`, not `null`/`None` — always check for it explicitly
4. `value` arrives as text and must be explicitly converted to numeric
5. Duplicates should be checked using `date` + `series_id` together, since the same date appears across every series
6. `reshape_data()` lets the caller choose between long and wide output depending on what they need downstream